# 00 · Setup & orientation

**The EDD demo in one sentence:** one EvalHub job spans behavioral + safety + adversarial evaluation, rolls up to **one verdict**, and lands as **one MLflow record** that gates a release (exit 0 = promote, exit 1 = blocked).

These notebooks run the **real** demo commands (`make` targets, `./release-gate.sh`) from cells — nothing is faked. Run them in order:

| Notebook | Demo |
|---|---|
| `00-setup.ipynb` | environment check + the modes you'll use |
| `01-agent-trace.ipynb` | **Demo 1** — the agent that *looks* fine but goes off-task |
| `02-release-gate-blocked.ipynb` | **Demo 2** — the gate blocks rc1 (exit 1) |
| `03-harden-and-promote.ipynb` | **Demo 3+4** — harden, then the gate promotes rc2 (exit 0) |
| `04-compliance-view.ipynb` | the MLflow compliance record (slide-16 payoff) |
| `05-live-maas.ipynb` | **online vs offline** — run the probe/gate against a live Red Hat MaaS model |

> Notebooks 00–04 run **fully offline** against the recorded fixtures in `./specs` — no docker, tokens, or network. `05` adds the **online** option (a live Red Hat AI MaaS endpoint) and shows how it degrades right back to offline if anything is missing.

### Step 1 — point the kernel at the repo root
Run this first in **every** notebook. It makes the demo commands resolve the same way they do from a terminal opened at the project root.

In [ ]:
import os
from pathlib import Path

# Notebooks live in <repo>/notebooks; walk up to the dir that holds the Makefile.
root = Path.cwd()
while not (root / "Makefile").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
print("working dir:", Path.cwd())

### Step 2 — check the environment
You need Python ≥ 3.11, `jq`, and the Python deps (`mlflow`, `requests`). `docker` is optional (only for the live path).

In [ ]:
%%bash
echo "python: $(python3 --version)"
command -v jq >/dev/null && echo "jq:     $(jq --version)" || echo "jq:     MISSING (brew/dnf install jq)"
command -v docker >/dev/null && echo "docker: $(docker --version)" || echo "docker: not installed (optional)"
python3 -c "import mlflow, requests; print('deps:   mlflow', mlflow.__version__, '+ requests OK')" 2>/dev/null || echo "deps:   MISSING -> run the next cell (make setup)"

### Step 3 — install deps (one time)
Only needed if the check above said deps are MISSING. Safe to skip otherwise.

In [ ]:
# Uncomment to install (mlflow + requests, editable install of this repo):
# !make setup

### The four gate modes
The release gate (and the harness behind it) selects how it answers via `GATE_MODE` or `--mode`. **Every live call degrades to the `./specs` fixtures on any error**, so a verdict is always produced.

| Mode | Behavior | When |
|---|---|---|
| `auto` *(default)* | EvalHub up → replay if a prebaked job exists else live; fixtures if down | on stage |
| `replay` | submit live, read the verdict from a prebaked completed job | on stage, after `make prebake` |
| `live` | real submit + poll to completion (slow: garak + lm-eval) | rehearsal |
| `offline` | fixtures only, zero network | **these notebooks** |

We use `--offline` throughout so the story is identical and instant. Move on to `01-agent-trace.ipynb`.